# Feature Engineering Exploration

This notebook covers advanced data exploration to support feature engineering, including:
1. **Long-tail Distributions** (User activity & Item popularity)
2. **Text Quality Checks** (After cleaning)
3. **Temporal Analysis** (Interaction trends)
4. **Feature Correlations**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

sns.set_style("whitegrid")
%matplotlib inline

## 1. Load Data

In [ ]:
# Load Item Metadata
books_df = pd.read_csv('../data/books_processed.csv')
print(f"Books loaded: {books_df.shape}")

# Load Interactions (Train)
# Using a sample if file is too large due to 1.2GB size, but let's try reading minimal columns first
train_df = pd.read_csv('../data/rec/train.csv', names=['isbn', 'user_id', 'rating', 'timestamp', 'review'], header=0)
# Note: header=0 because train.csv usually has header. Let's check head first usually, but assuming standard format.
print(f"Interactions loaded: {train_df.shape}")

In [ ]:
train_df.head()

## 2. Long-tail Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# User Activity
user_counts = train_df['user_id'].value_counts().values
sns.histplot(user_counts, bins=50, log_scale=(True, True), ax=axes[0])
axes[0].set_title("User Interaction Distribution (Log-Log)")
axes[0].set_xlabel("Number of Ratings (Log)")
axes[0].set_ylabel("Count of Users (Log)")

# Item Popularity
item_counts = train_df['isbn'].value_counts().values
sns.histplot(item_counts, bins=50, log_scale=(True, True), ax=axes[1])
axes[1].set_title("Item Popularity Distribution (Log-Log)")
axes[1].set_xlabel("Number of Ratings (Log)")
axes[1].set_ylabel("Count of Books (Log)")

plt.show()

In [ ]:
print(f"Max ratings by user: {user_counts.max()}")
print(f"Median ratings by user: {np.median(user_counts)}")
print(f"Max ratings per book: {item_counts.max()}")
print(f"Median ratings per book: {np.median(item_counts)}")

# Sparsity
n_users = train_df['user_id'].nunique()
n_items = train_df['isbn'].nunique()
n_ratings = len(train_df)
sparsity = 1 - (n_ratings / (n_users * n_items))
print(f"Sparsity: {sparsity:.6f}")

## 3. Text Quality Analysis

In [ ]:
# Check Description Lengths
books_df['desc_len'] = books_df['description'].fillna("").apply(len)

plt.figure(figsize=(10, 5))
sns.histplot(books_df['desc_len'], bins=50)
plt.title("Description Length Distribution")
plt.show()

In [ ]:
# Check for HTML tag artifacts
def has_html(text):
    if not isinstance(text, str): return False
    return bool(re.search(r'<[^>]+>', text))

books_df['has_html'] = books_df['description'].apply(has_html)
html_count = books_df['has_html'].sum()
print(f"Rows with potential HTML tags in description: {html_count} / {len(books_df)}")

if html_count > 0:
    print(books_df[books_df['has_html']]['description'].iloc[0])

In [ ]:
# Compare Description Length vs. Rating Count (Popularity)
# Merge count info
item_pop = train_df['isbn'].value_counts().reset_index()
item_pop.columns = ['isbn13', 'rating_count'] # assuming isbn match

# Ensure convert to string for merge
books_df['isbn13'] = books_df['isbn13'].astype(str).str.replace(r'\.0$', '', regex=True)
item_pop['isbn13'] = item_pop['isbn13'].astype(str)

merged = pd.merge(books_df, item_pop, on='isbn13', how='inner')

plt.figure(figsize=(8, 6))
sns.scatterplot(data=merged, x='desc_len', y='rating_count', alpha=0.3)
plt.yscale('log')
plt.title("Description Length vs. Popularity")
plt.show()

## 4. Temporal Analysis

In [ ]:
train_df['date'] = pd.to_datetime(train_df['timestamp'], unit='s', errors='coerce')

# Ratings over time
daily_counts = train_df.set_index('date').resample('M')['rating'].count()

plt.figure(figsize=(14, 6))
daily_counts.plot()
plt.title("Monthly Interaction Volume")
plt.ylabel("Number of Ratings")
plt.show()

In [ ]:
# Average Rating over time
daily_avg = train_df.set_index('date').resample('M')['rating'].mean()

plt.figure(figsize=(14, 6))
daily_avg.plot(color='orange')
plt.title("Monthly Average Rating Trend")
plt.ylabel("Average Rating")
plt.ylim(1, 5)
plt.show()